In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# 1. Install YOLOv8
!pip install ultralytics
!pip install albumentations  # Useful for image augmentations

import os
import json
import cv2
import numpy as np
import pandas as pd
import glob
import random
import shutil
from tqdm.notebook import tqdm
from ultralytics import YOLO
import matplotlib.pyplot as plt

# 2. Define EXACT Paths based on your output
BASE_PATH = "/kaggle/input/vista26/Vistas Dataset Public"
TRAIN_IMG_DIR = os.path.join(BASE_PATH, "train")
TEST_IMG_DIR = os.path.join(BASE_PATH, "test")
BG_DIR = os.path.join(BASE_PATH, "background")  # Using the real background files
ANNOTATIONS_PATH = os.path.join(BASE_PATH, "instances_train.json")
CATEGORIES_PATH = os.path.join(BASE_PATH, "Categories.json")

# 3. Create Working Directories
WORK_DIR = "/kaggle/working"
SYNTHETIC_DIR = os.path.join(WORK_DIR, "synthetic_dataset")

# Clean up if exists to avoid errors on re-run
if os.path.exists(SYNTHETIC_DIR):
    shutil.rmtree(SYNTHETIC_DIR)

for p in ['images/train', 'images/val', 'labels/train', 'labels/val']:
    os.makedirs(os.path.join(SYNTHETIC_DIR, p), exist_ok=True)
    
print("Environment Ready. YOLO installed and directories created.")

In [ ]:
import json
import cv2
import os
import glob
import numpy as np
import random
from tqdm.notebook import tqdm

print("Loading metadata...")

# 1. Load Categories separately (Fix for KeyError)
with open(CATEGORIES_PATH, 'r') as f:
    cat_raw = json.load(f)

# Check if Categories.json is a list or a dict
if isinstance(cat_raw, dict) and 'categories' in cat_raw:
    categories = cat_raw['categories']
elif isinstance(cat_raw, list):
    categories = cat_raw
else:
    # Fallback: sometimes it's just a dict of ID->Name
    categories = [{'id': k, 'name': v} for k, v in cat_raw.items()]

# Sort and Map IDs
categories = sorted(categories, key=lambda x: x['id'])
cat_id_to_yolo = {cat['id']: i for i, cat in enumerate(categories)}
yolo_to_cat_id = {i: cat['id'] for i, cat in enumerate(categories)}
class_names = [cat['name'] for cat in categories]

print(f"Loaded {len(categories)} categories.")

# 2. Load Train Annotations
with open(ANNOTATIONS_PATH, 'r') as f:
    train_data = json.load(f)

# Index images and annotations
images_info = {img['id']: img for img in train_data['images']}
img_to_anns = {}
for ann in train_data['annotations']:
    img_to_anns.setdefault(ann['image_id'], []).append(ann)

# 3. Load Background Images
bg_files = glob.glob(os.path.join(BG_DIR, "*"))
backgrounds = []
for f in bg_files:
    img = cv2.imread(f)
    if img is not None:
        backgrounds.append(img)
        
print(f"Loaded {len(backgrounds)} background templates.")

# --- HELPER FUNCTIONS ---
def get_crop(img_id):
    """Crops the object from the source training image."""
    if img_id not in images_info: return None, None
    img_info = images_info[img_id]
    
    # Construct path
    img_path = os.path.join(TRAIN_IMG_DIR, img_info['file_name'])
    if not os.path.exists(img_path): return None, None
    
    img = cv2.imread(img_path)
    if img is None: return None, None
    
    # Get annotation (assuming single object per training image)
    if img_id not in img_to_anns: return None, None
    ann = img_to_anns[img_id][0]
    x, y, w, h = map(int, ann['bbox'])
    
    # Safe Crop logic
    h_img, w_img = img.shape[:2]
    x, y = max(0, x), max(0, y)
    w = min(w, w_img - x)
    h = min(h, h_img - y)
    
    if w <= 0 or h <= 0: return None, None
    
    crop = img[y:y+h, x:x+w]
    return crop, ann['category_id']

# --- GENERATION LOOP ---
NUM_SYNTH_IMAGES = 1500  
IMG_SIZE = 640
valid_ids = list(images_info.keys())

print(f"Generating {NUM_SYNTH_IMAGES} synthetic images...")

for i in tqdm(range(NUM_SYNTH_IMAGES)):
    # Pick Background
    if backgrounds:
        bg_base = random.choice(backgrounds).copy()
        bg_base = cv2.resize(bg_base, (IMG_SIZE, IMG_SIZE))
    else:
        bg_base = np.full((IMG_SIZE, IMG_SIZE, 3), 200, dtype=np.uint8)

    labels = []
    
    # Randomly paste 3 to 8 objects
    num_objs = random.randint(3, 8)
    
    for _ in range(num_objs):
        rand_id = random.choice(valid_ids)
        crop, cat_id = get_crop(rand_id)
        
        if crop is None: continue
        
        # Resize crop
        h, w = crop.shape[:2]
        scale = random.uniform(0.3, 0.8) 
        new_w, new_h = int(w * scale), int(h * scale)
        if new_w < 10 or new_h < 10: continue 
        
        crop_resized = cv2.resize(crop, (new_w, new_h))
        
        # Random Position
        if IMG_SIZE - new_w <= 0 or IMG_SIZE - new_h <= 0: continue
        x_off = random.randint(0, IMG_SIZE - new_w)
        y_off = random.randint(0, IMG_SIZE - new_h)
        
        # Paste
        bg_base[y_off:y_off+new_h, x_off:x_off+new_w] = crop_resized
        
        # YOLO Coordinates
        x_c = (x_off + new_w / 2) / IMG_SIZE
        y_c = (y_off + new_h / 2) / IMG_SIZE
        w_n = new_w / IMG_SIZE
        h_n = new_h / IMG_SIZE
        
        labels.append(f"{cat_id_to_yolo[cat_id]} {x_c:.6f} {y_c:.6f} {w_n:.6f} {h_n:.6f}")
    
    # Save
    subset = 'train' if i < NUM_SYNTH_IMAGES * 0.8 else 'val'
    fname = f"syn_{i:05d}"
    
    cv2.imwrite(os.path.join(SYNTHETIC_DIR, f'images/{subset}/{fname}.jpg'), bg_base)
    with open(os.path.join(SYNTHETIC_DIR, f'labels/{subset}/{fname}.txt'), 'w') as f:
        f.write('\n'.join(labels))

print("Step 2 Complete: Synthetic Data Generated.")

In [ ]:
# ==============================
# STEP 3: Train Model & Submit 
# ==============================
import pandas as pd
import glob
import os
import json
from ultralytics import YOLO
from tqdm.notebook import tqdm

# --- CONFIGURATION ---
WORK_DIR = "/kaggle/working"
TEST_IMG_DIR = "/kaggle/input/vista26/Vistas Dataset Public/test"
CATEGORIES_PATH = "/kaggle/input/vista26/Vistas Dataset Public/Categories.json"
SYNTHETIC_DIR = os.path.join(WORK_DIR, "synthetic_dataset")

# 1. Ensure Class Names are Loaded
# (In case you restarted the kernel, we reload them here)
print("Verifying Class Names...")
with open(CATEGORIES_PATH, 'r') as f:
    cat_raw = json.load(f)
    if isinstance(cat_raw, dict) and 'categories' in cat_raw: cats = cat_raw['categories']
    elif isinstance(cat_raw, list): cats = cat_raw
    else: cats = [{'id': k, 'name': v} for k, v in cat_raw.items()]
    
    # Sort by ID to ensure consistent mapping
    cats = sorted(cats, key=lambda x: x['id'])
    class_names = [c['name'] for c in cats]
    # Create mapping: YOLO Index -> Real Category ID
    yolo_to_cat_id = {i: c['id'] for i, c in enumerate(cats)}

# 2. Create Data YAML
yaml_content = f"""
path: {SYNTHETIC_DIR}
train: images/train
val: images/val
names: {class_names}
"""
with open("vista.yaml", "w") as f:
    f.write(yaml_content)

# 3. Train YOLOv8
print("Starting Training...")
model = YOLO("yolov8s.pt") 

model.train(
    data="vista.yaml",
    epochs=10, 
    imgsz=640,
    batch=16,
    project="vista_cv", 
    name="submission_run",
    verbose=True,
    exist_ok=True
)

# 4. AUTOMATIC WEIGHT FINDER (The Fix)
print("Searching for trained model weights...")
# Look for best.pt anywhere in the working directory
possible_weights = glob.glob(os.path.join(WORK_DIR, "**", "best.pt"), recursive=True)

if not possible_weights:
    raise FileNotFoundError("Could not find 'best.pt'. Training might have failed.")

# Pick the most recent one
best_weight_path = possible_weights[-1] 
print(f"FOUND WEIGHTS AT: {best_weight_path}")

# 5. Inference
best_model = YOLO(best_weight_path)

# Locate Test Images
test_files = glob.glob(os.path.join(TEST_IMG_DIR, "*.jpg"))
if not test_files:
    print("Test folder empty! Using Validation folder for demo purposes.")
    test_files = glob.glob("/kaggle/input/vista26/Vistas Dataset Public/validation/*.jpg")

print(f"Predicting on {len(test_files)} images...")

submission_rows = []
CONF_THRESHOLD = 0.50 

for img_path in tqdm(test_files):
    img_id = os.path.basename(img_path).split('.')[0]
    
    results = best_model.predict(img_path, conf=CONF_THRESHOLD, verbose=False)[0]
    
    preds_str = []
    for box in results.boxes:
        cls_idx = int(box.cls[0])
        # Map back to Real Category ID
        real_cat_id = yolo_to_cat_id.get(cls_idx, cls_idx)
        
        conf = float(box.conf[0])
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        
        # Format: CategoryID Conf Xmin Ymin Xmax Ymax
        preds_str.append(f"{real_cat_id} {conf:.4f} {x1:.2f} {y1:.2f} {x2:.2f} {y2:.2f}")
        
    submission_rows.append({
        "ImageID": img_id,
        "PredictionString": " ".join(preds_str)
    })

# 6. Save CSV
df = pd.DataFrame(submission_rows)
df = df[['ImageID', 'PredictionString']] 
df.to_csv("submission.csv", index=False)

print(f"SUCCESS: submission.csv generated with {len(df)} rows.")